# Data Ingestion


###  Document Structure

In [ ]:


from langchain_core.documents import Document

In [6]:
doc = Document(
    page_content = " this is the text content I am using to create RAG"
,   metadata = {
    "source" : "example.txt",
    "pages" : 1,
    "author": "Adarsh Chaubey",
    "data_created": "2025-01-01"    
}
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Adarsh Chaubey', 'data_created': '2025-01-01'}, page_content=' this is the text content I am using to create RAG')

### Create a simple txt file 

In [ ]:

import os 
os.makedirs("../data/txt_files",exist_ok=True)

In [11]:
sample_texts = {
    "../data/txt_files/python_intro.txt": "Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python has a large standard library and a vibrant ecosystem of third-party packages, making it suitable for web development, data analysis, artificial intelligence, scientific computing, and more.",
}

for filepath, content in sample_texts.items():
    with open(filepath,"w",encoding="utf-8") as f:
        f.write(content)
        
print("Sample text files created successfully");

Sample text files created successfully


### TextLoader

In [17]:


from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    "../data/txt_files/python_intro.txt",
    encoding="utf-8"
)

document = loader.load()
print(document)

[Document(metadata={'source': '../data/txt_files/python_intro.txt'}, page_content='Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python has a large standard library and a vibrant ecosystem of third-party packages, making it suitable for web development, data analysis, artificial intelligence, scientific computing, and more.')]


### Directory Loader

In [34]:


from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Load all text files in a directory
dir_loader=DirectoryLoader(
    "../data/txt_files",
    glob="*.txt", # pattern to match files, e.g., "*.txt" for text files
    loader_cls=TextLoader, # class of the loader to use
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)

documents = dir_loader.load()
print(documents)

100%|██████████| 2/2 [00:00<00:00, 152.66it/s]

[Document(metadata={'source': '..\\data\\txt_files\\python.txt'}, page_content=''), Document(metadata={'source': '..\\data\\txt_files\\python_intro.txt'}, page_content='Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python has a large standard library and a vibrant ecosystem of third-party packages, making it suitable for web development, data analysis, artificial intelligence, scientific computing, and more.')]


In [35]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="*.pdf", # pattern to match files, e.g., "*.txt" for text files
    loader_cls=PyMuPDFLoader, # class of the loader to use
    show_progress=False
)
pdf_doc = dir_loader.load()
pdf_doc

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': '', 'creationdate': '2026-10-01T21:21:18+05:30', 'source': '..\\data\\pdf\\RAG11.pdf', 'file_path': '..\\data\\pdf\\RAG11.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': 'Microsoft Word - Document1', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-10-01T21:21:18+05:30', 'trapped': '', 'modDate': "D:20261001212118+05'30'", 'creationDate': "D:20261001212118+05'30'", 'page': 0}, page_content='')]

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Chunking the documents

In [36]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split non-empty documents into smaller chunks for better RAG performance."""
    non_empty_documents = [
        document
        for document in documents
        if document.page_content and document.page_content.strip()
    ]

    if not non_empty_documents:
        raise ValueError(
            "No extractable text was found. Add text to the source PDF or provide a text document."
        )

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(non_empty_documents)

    print(
        f"Split {len(non_empty_documents)} documents into "
        f"{len(split_docs)} chunks"
    )

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(
            f"Content: {split_docs[0].page_content[:200]}..."
        )
        print(
            f"Metadata: {split_docs[0].metadata}"
        )

    return split_docs


# Use PDF text when available; otherwise use the loaded text files.
source_documents = pdf_doc
if not any(document.page_content and document.page_content.strip() for document in source_documents):
    print("No text found in the PDF; falling back to text documents.")
    source_documents = documents

chunks = split_documents(source_documents)

chunks

Split 1 documents into 0 chunks


[]

### Embedding the documents

In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initializes the embedding manager with a specified model

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads the SentenceTransformer model"""
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model {self.model_name} loaded successfully. "
                f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generates embeddings for a list of texts

        Args:
            texts: List of strings to embed

        Returns:
            np.ndarray: Array of embeddings
        """
        if self.model is None:
            raise ValueError("Model Not Loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embedding = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings of shape: {embedding.shape}")
        return embedding


# Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading model: all-MiniLM-L6-v2


c:\Users\amitp\OneDrive\Desktop\Code ML\RAG Pipelines\RAG-DataIngestionPipeline\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\amitp\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\amitp\OneDrive\Desktop\Code ML\RAG 

Model all-MiniLM-L6-v2 loaded successfully. Embedding dimension: 384


C:\Users\amitp\AppData\Local\Temp\ipykernel_21184\1397675924.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"


### Vector Store

In [19]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        """Initializes the vector store

        Args:
            collection_name: Name of the collection in ChromaDB
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initializes the ChromaDB client and collection"""
        try:
            print(
                f"Initializing ChromaDB client with persist directory: "
                f"{self.persist_directory}"
            )

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Collection of PDF document embeddings"
                }
            )

            print(
                f"Collection '{self.collection_name}' initialized successfully."
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """Adds documents and their embeddings to the vector store

        Args:
            documents: List of document objects containing page_content
                       and metadata.
            embeddings: NumPy array containing document embeddings.
        """

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents and embeddings must be the same."
            )

        print(
            f"Adding {len(documents)} documents to vector store...."
        )

        # Prepare data for ChromaDB
        ids = []
        embeddings_list = []
        metadatas = []
        documents_text = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):
            doc_id = f"doc_{i}_{uuid.uuid4().hex[:8]}"

            ids.append(doc_id)
            embeddings_list.append(embedding.tolist())
            metadatas.append(doc.metadata)
            documents_text.append(doc.page_content)

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Added {len(documents)} documents to the collection "
                f"'{self.collection_name}'."
            )

            print(
                f"Total documents in collection after addition: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise


# Initialize the vector store
vector_store = VectorStore()

vector_store

Initializing ChromaDB client with persist directory: ../data/vector_store
Collection 'pdf_documents' initialized successfully.
Existing documents in collection: 0


In [38]:
chunks

[]